# 第3回：pandasで表データに触る

**今日の問い：初めて見る表データを受け取ったら、最初に何を見るか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 初見データの形・型・欠損・要約統計を確認する
- locとqueryで条件を明示し、method chainingで読みやすくまとめる
- groupby・agg・pivot_tableで多軸の比較表を作り、性能差にも気を配る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- DataFrame：行と列を持つ表
- method chaining：中間変数を作らず処理をつなげる書き方
- ベクトル化：ループの代わりに列全体へ一括演算すること
- 集約：複数行を件数や平均などへまとめる処理
- カテゴリ型：取りうる値が限られる列の省メモリ表現

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：最初の健康診断

形・型・欠損・ユニーク数・要約統計を1度に確認します。


In [ ]:
print("形:", df.shape)
quality = pd.DataFrame({
    "データ型": df.dtypes.astype(str),
    "欠損数": df.isna().sum(),
    "欠損率": df.isna().mean().round(3),
    "ユニーク数": df.nunique(),
})
display(quality)
display(df.select_dtypes(include="number").describe().T.round(2))


## 行と列を選ぶ（locとquery）

条件を文字列で書けるqueryは、複数条件を読みやすくします。


In [ ]:
columns = ["sample_id", "solvent", "catalyst", "temperature_c", "yield_pct", "active"]
display(df.loc[:4, columns])
subset = df.query("catalyst == 'Cat-A' and temperature_c >= 80")[columns]
print("Cat-Aかつ80℃以上:", len(subset), "件")
subset.head()


## TRY：カテゴリごとに比べる

平均だけでなく件数とばらつきも一緒に見ます。


In [ ]:
solvent_summary = (
    df.groupby("solvent", dropna=False)
      .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"),
           収率SD=("yield_pct", "std"), 活性率=("active", "mean"))
      .sort_values("平均収率", ascending=False)
)
solvent_summary.round(2)


## CHANGE

`solvent`を`catalyst`や`scaffold_group`へ変えます。順位が変わる理由は、データだけから断定せず仮説として書きます。


## DEEP DIVE：pivot_table・pipe・ベクトル化

多軸の集計、処理をつなぐ書き方、速度の3点を扱います。


In [ ]:
pivot = pd.pivot_table(df, index="catalyst", columns="solvent", values="yield_pct", aggfunc=["count", "mean"])
pivot.round(1)


### pipeで処理を関数としてつなぐ

中間変数を増やさず、意図を関数名で表せます（元データは変更しない）。


In [ ]:
def add_quality_flags(frame):
    "収率の中央値以上かどうかのフラグ列を足して返す（元は変更しない）。"
    out = frame.copy()
    out["high_yield"] = out["yield_pct"] >= out["yield_pct"].median()
    return out

summary = (
    df
    .pipe(add_quality_flags)
    .groupby(["catalyst", "high_yield"], observed=True)
    .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"))
    .round(2)
)
summary


## CHALLENGE：applyとベクトル化の速度差

行ごとのapplyは読みやすい反面、遅くなりがちです。同じ結果をベクトル化で書き、時間を比べます。


In [ ]:
import time

def slow_flag(frame):
    return frame.apply(lambda r: r["temperature_c"] >= 80 and r["catalyst"] == "Cat-A", axis=1)

def fast_flag(frame):
    return (frame["temperature_c"] >= 80) & (frame["catalyst"] == "Cat-A")

t0 = time.perf_counter(); a = slow_flag(df); t1 = time.perf_counter()
b = fast_flag(df); t2 = time.perf_counter()
print("apply     :", round((t1 - t0) * 1000, 2), "ms")
print("vectorized:", round((t2 - t1) * 1000, 2), "ms")
print("結果一致:", bool((a.fillna(False) == b.fillna(False)).all()))


## よくある誤り

- 列の単位や定義を確認せず計算する
- 行ごとのapplyを多用して遅く読みにくくする
- 件数が極端に少ない群の平均を強く信じる

## SELF-STUDY（任意・30〜60分）

- 触媒×溶媒の件数・平均収率・標準偏差をpivot_tableで作る
- applyとベクトル化の実行時間を比較し、差をm%で記録する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. shapeの2つの数は何か
2. method chainingの利点と注意点は何か
3. applyよりベクトル化を選ぶ理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
